# Arabic Sign Language (ArSL) Translation

This project aims to build an AI model that recognizes Arabic Sign Language gestures via a live camera and translates them into text in real time, using **MediaPipe** to extract body and hand landmarks, and a **BiLSTM** model trained on the **KArSL** dataset

## 1. Setup Libraries + Google Drive

In [1]:
# Install the required libraries
!pip install py7zr mediapipe -q

import os
import glob
import gc
import multiprocessing
import json

import cv2
import numpy as np
import py7zr
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm
from google.colab import drive

drive.mount('/content/drive')

# Drive Path
BASE_DRIVE_PATH = '/content/drive/MyDrive/KArSL_Project'
LOCAL_RAW_PATH = '/content/karsl_raw'
PROCESSED_SAVE_PATH = f'{BASE_DRIVE_PATH}/processed_landmarks'
MODEL_SAVE_PATH = f'{BASE_DRIVE_PATH}/sign_lstm_model_best.pth'
IDX_TO_CLASS_PATH = f'{BASE_DRIVE_PATH}/idx_to_class.json'

TARGET_SEQ_LEN = 30          # Number of frames per sign after resampling (resampling)
NUM_LANDMARKS = 33 + 21 + 21  # pose + left hand + right hand
INPUT_DIM = NUM_LANDMARKS * 3  # = 225 (x, y, z  per point)

os.makedirs(LOCAL_RAW_PATH, exist_ok=True)
os.makedirs(PROCESSED_SAVE_PATH, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[INFO] Using device: {device}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.9/37.9 MB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 492.7/492.7 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.5/144.5 kB 6.9 MB/s eta 0:00:00
Mounted at /content/drive
[INFO] Using device: cpu


## 2. Unzip raw data archives (KArSL)

In [2]:
archive_files = sorted(glob.glob(f'{BASE_DRIVE_PATH}/train_0*/*.7z'))
print(f"[INFO] Found {len(archive_files)} .7z archives in Google Drive.")

for archive in archive_files:
    archive_name = os.path.basename(archive)
    print(f"[EXTRACTING] {archive_name}...")
    try:
        with py7zr.SevenZipFile(archive, mode='r') as z:
            z.extractall(path=LOCAL_RAW_PATH)
        print(f"[SUCCESS] Extracted: {archive_name}")
    except Exception as e:
        print(f"[ERROR] Failed to extract {archive_name}: {e}")

print("\n[SUCCESS] Extraction completed to local storage.")

[INFO] Found 9 .7z archives in Google Drive.
[EXTRACTING] 0001-0071.7z...
[SUCCESS] Extracted: 0001-0071.7z
[EXTRACTING] 0071-0170.7z...
[SUCCESS] Extracted: 0071-0170.7z
[EXTRACTING] 0171-0502.7z...
[SUCCESS] Extracted: 0171-0502.7z
[EXTRACTING] 0001-0070.7z...
[SUCCESS] Extracted: 0001-0070.7z
[EXTRACTING] 0071-0170.7z...
[SUCCESS] Extracted: 0071-0170.7z
[EXTRACTING] 0171-0502.7z...
[SUCCESS] Extracted: 0171-0502.7z
[EXTRACTING] 0001-0070.7z...
[SUCCESS] Extracted: 0001-0070.7z
[EXTRACTING] 0071-0170.7z...
[SUCCESS] Extracted: 0071-0170.7z
[EXTRACTING] 0171-0502.7z...
[SUCCESS] Extracted: 0171-0502.7z

[SUCCESS] Extraction completed to local storage.


## 3.  Load MediaPipe (Pose + Hand Landmarker) Model

In [3]:
!wget -q -O pose_landmarker.task https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_heavy/float16/latest/pose_landmarker_heavy.task
!wget -q -O hand_landmarker.task https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task

## 4. Extract The Landmarks Frim Each Video

In [4]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

POSE_MODEL_PATH = 'pose_landmarker.task'
HAND_MODEL_PATH = 'hand_landmarker.task'

pose_options = vision.PoseLandmarkerOptions(
    base_options=python.BaseOptions(model_asset_path=POSE_MODEL_PATH),
    running_mode=vision.RunningMode.IMAGE,
    min_pose_detection_confidence=0.5,
    min_tracking_confidence=0.5,
)

hand_options = vision.HandLandmarkerOptions(
    base_options=python.BaseOptions(model_asset_path=HAND_MODEL_PATH),
    running_mode=vision.RunningMode.IMAGE,
    num_hands=2,
    min_hand_detection_confidence=0.5,
    min_tracking_confidence=0.5,
)


def extract_normalized_landmarks(pose_result, hand_result):
    """
    Transform the resuls to single vector printed with lenght of 225 pose + Hand  lanmarker.
    """
    if not pose_result.pose_landmarks or len(pose_result.pose_landmarks) == 0:
        return np.zeros(INPUT_DIM, dtype=np.float32)

    pose_pts = np.array([[lm.x, lm.y, lm.z] for lm in pose_result.pose_landmarks[0]], dtype=np.float32)
    left_shoulder, right_shoulder = pose_pts[11], pose_pts[12]
    center = (left_shoulder + right_shoulder) / 2.0
    shoulder_dist = np.linalg.norm(left_shoulder - right_shoulder) or 1.0

    norm_pose = (pose_pts - center) / shoulder_dist
    norm_lh = np.zeros((21, 3), dtype=np.float32)
    norm_rh = np.zeros((21, 3), dtype=np.float32)

    if hand_result.hand_landmarks and hand_result.handedness:
        for hand_lms, handedness in zip(hand_result.hand_landmarks, hand_result.handedness):
            pts = np.array([[lm.x, lm.y, lm.z] for lm in hand_lms], dtype=np.float32)
            norm_pts = (pts - center) / shoulder_dist
            if handedness[0].category_name == 'Left':
                norm_lh = norm_pts
            else:
                norm_rh = norm_pts

    return np.concatenate([norm_pose.flatten(), norm_lh.flatten(), norm_rh.flatten()])


_worker_pose_detector = None
_worker_hand_detector = None


def _init_worker():
    """
    It is called only once when each parallel worker process starts creating
    MediaPipe detectors once per process rather than recreating them for each class (which is much faster).
    """
    global _worker_pose_detector, _worker_hand_detector
    _worker_pose_detector = vision.PoseLandmarker.create_from_options(pose_options)
    _worker_hand_detector = vision.HandLandmarker.create_from_options(hand_options)


def process_class(class_name):
    """
    Videos of an entire class are processed using current-operation detectors.
    """
    pose_detector = _worker_pose_detector
    hand_detector = _worker_hand_detector

    local_class_dir = os.path.join(LOCAL_RAW_PATH, class_name)
    save_class_dir = os.path.join(PROCESSED_SAVE_PATH, class_name)
    os.makedirs(save_class_dir, exist_ok=True)

    sample_names = sorted([
        s for s in os.listdir(local_class_dir)
        if os.path.isdir(os.path.join(local_class_dir, s))
    ])

    for sample_id in sample_names:
        save_file_path = os.path.join(save_class_dir, f"{sample_id}.npy")
        if os.path.exists(save_file_path):
            continue

        local_sample_dir = os.path.join(local_class_dir, sample_id)
        jpg_frames = sorted([
            os.path.join(local_sample_dir, f) for f in os.listdir(local_sample_dir)
            if f.lower().endswith('.jpg')
        ])
        if not jpg_frames:
            continue

        sequence_landmarks = []
        for frame_path in jpg_frames:
            frame = cv2.imread(frame_path)
            if frame is None:
                continue
            frame = cv2.resize(frame, (256, 256))
            image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)

            pose_result = pose_detector.detect(mp_image)
            hand_result = hand_detector.detect(mp_image)
            sequence_landmarks.append(extract_normalized_landmarks(pose_result, hand_result))

        if sequence_landmarks:
            indices = np.linspace(0, len(sequence_landmarks) - 1, TARGET_SEQ_LEN, dtype=int)
            resampled = np.array(sequence_landmarks)[indices]
            np.save(save_file_path, resampled)

        del sequence_landmarks

    gc.collect()
    return class_name


class_names = sorted([
    d for d in os.listdir(LOCAL_RAW_PATH)
    if os.path.isdir(os.path.join(LOCAL_RAW_PATH, d)) and d.isdigit() and len(d) == 4
])

NUM_WORKERS = os.cpu_count() or 2
print(f"[INFO] Processing {len(class_names)} classes across {NUM_WORKERS} parallel workers...")

with multiprocessing.Pool(processes=NUM_WORKERS, initializer=_init_worker) as pool:
    for _ in tqdm(pool.imap_unordered(process_class, class_names), total=len(class_names), desc="Processing Classes"):
        pass

print("\n[COMPLETE] Done processing all classes.")

[INFO] Processing 502 classes across 2 parallel workers...


Processing Classes:  61%|██████▏   | 308/502 [24:22<15:20,  4.75s/it]    


KeyboardInterrupt: 

## 5. The Data Model

Only a single clean copy (instead of the three consecutive copies). Augmentation is applied only
to the training set via a separate `AugmentedSubset` class without affecting the validation data.

In [5]:
class KArSLInMemoryDataset(Dataset):

    def __init__(self, data_dir, oversample_custom_factor=1, allowed_classes=None):
        self.samples, self.labels = [], []

        class_dirs = sorted([d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))])
        if allowed_classes is not None:
            class_dirs = [d for d in class_dirs if d in allowed_classes]
        self.class_to_idx = {c_name: idx for idx, c_name in enumerate(class_dirs)}

        print("[INFO] Loading all data into RAM...")
        custom_count = 0
        for c_name in class_dirs:
            c_path = os.path.join(data_dir, c_name)
            c_idx = self.class_to_idx[c_name]
            for file_name in os.listdir(c_path):
                if not file_name.endswith('.npy'):
                    continue
                data = np.load(os.path.join(c_path, file_name))
                repeats = oversample_custom_factor if file_name.startswith('custom_') else 1
                if repeats > 1:
                    custom_count += 1
                for _ in range(repeats):
                    self.samples.append(data)
                    self.labels.append(c_idx)

        print(f"[INFO] Loaded {len(self.samples)} samples from {len(class_dirs)} classes "
              f"({custom_count} custom samples, Each one is aggregated x{oversample_custom_factor}).")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sequence = self.samples[idx].copy()
        return torch.tensor(sequence, dtype=torch.float32), torch.tensor(self.labels[idx], dtype=torch.long)


def geometric_augment(sequence):

    points = sequence.reshape(sequence.shape[0], -1, 3).copy()  # (T, 75, 3)

# 1. Simple rotation around the Z-axis (simulating camera angle variation) — from -15 to 15 degrees
    angle = np.radians(np.random.uniform(-15, 15))
    cos_a, sin_a = np.cos(angle), np.sin(angle)
    x, y = points[..., 0].copy(), points[..., 1].copy()
    points[..., 0] = x * cos_a - y * sin_a
    points[..., 1] = x * sin_a + y * cos_a

# 2. Random scaling (simulates varying distances from the camera)
    scale = np.random.uniform(0.85, 1.15)
    points *= scale

# 3. Slight offset (simulates a person standing slightly off-center in the frame)
    shift_x = np.random.uniform(-0.05, 0.05)
    shift_y = np.random.uniform(-0.05, 0.05)
    points[..., 0] += shift_x
    points[..., 1] += shift_y

    return points.reshape(sequence.shape).astype(np.float32)


class AugmentedSubset(Dataset):

    def __init__(self, base_dataset, indices, augment=False):
        self.base_dataset = base_dataset
        self.indices = indices
        self.augment = augment

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        sequence = self.base_dataset.samples[real_idx].copy()
        label = self.base_dataset.labels[real_idx]

        if self.augment:
           # 1. Geometric transformation (rotation + scaling + translation).
            if np.random.rand() > 0.4:
                sequence = geometric_augment(sequence)

          # 2. Very slight joint jitter (Jittering)
            if np.random.rand() > 0.5:
                sequence = sequence + np.random.normal(0, 0.01, sequence.shape).astype(np.float32)

          # 3. Randomly drop one or two frames and then resample (to simulate speed variation)
            if np.random.rand() > 0.7 and len(sequence) > 10:
                drop_idx = np.random.choice(len(sequence), size=2, replace=False)
                sequence = np.delete(sequence, drop_idx, axis=0)
                resample_idx = np.linspace(0, len(sequence) - 1, TARGET_SEQ_LEN, dtype=int)
                sequence = sequence[resample_idx]

        return torch.tensor(sequence, dtype=torch.float32), torch.tensor(label, dtype=torch.long)


class SignLSTM(nn.Module):

    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True,
                             dropout=0.3, bidirectional=True)
        # hidden_dim * 2 (bidirectional) * 2 (mean pooling + max pooling )
        self.fc = nn.Linear(hidden_dim * 2 * 2, output_dim)

    def forward(self, x):
        out, _ = self.lstm(x)                    # (Batch, Seq_Len, hidden_dim * 2)
        mean_pool = out.mean(dim=1)
        max_pool, _ = out.max(dim=1)
        combined = torch.cat([mean_pool, max_pool], dim=1)
        return self.fc(combined)

## 6. Item Dictionary + Best Checkpoint Training With Best Model Saving

In [6]:
full_dataset = KArSLInMemoryDataset(PROCESSED_SAVE_PATH)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
generator = torch.Generator().manual_seed(42)
train_subset, val_subset = random_split(full_dataset, [train_size, val_size], generator=generator)

train_dataset = AugmentedSubset(full_dataset, train_subset.indices, augment=True)
val_dataset = AugmentedSubset(full_dataset, val_subset.indices, augment=False)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)

print(f"[INFO] Training samples: {len(train_dataset)} | Validation samples: {len(val_dataset)}")

idx_to_class = {idx: c_name for c_name, idx in full_dataset.class_to_idx.items()}
with open(IDX_TO_CLASS_PATH, 'w', encoding='utf-8') as f:
    json.dump(idx_to_class, f, ensure_ascii=False, indent=2)
print(f"[INFO] Saved idx_to_class mapping -> {IDX_TO_CLASS_PATH}")

model = SignLSTM(INPUT_DIM, hidden_dim=128, output_dim=len(full_dataset.class_to_idx)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

num_epochs = 30
best_val_acc = 0.0

for epoch in range(num_epochs):

    # Train
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    for sequences, labels in train_loader:
        sequences, labels = sequences.to(device), labels.to(device)

        outputs = model(sequences)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * sequences.size(0)
        train_correct += (outputs.argmax(1) == labels).sum().item()
        train_total += labels.size(0)

    # validation
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for sequences, labels in val_loader:
            sequences, labels = sequences.to(device), labels.to(device)
            outputs = model(sequences)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * sequences.size(0)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += labels.size(0)

    train_acc = train_correct / train_total
    val_acc = val_correct / val_total
    scheduler.step(val_acc)

    print(f"Epoch [{epoch+1}/{num_epochs}] | "
          f"Train Loss: {train_loss/train_total:.4f} Acc: {train_acc*100:.2f}% || "
          f"Val Loss: {val_loss/val_total:.4f} Acc: {val_acc*100:.2f}%")

    # save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'model_state_dict': model.state_dict(),
            'input_dim': INPUT_DIM,
            'hidden_dim': 128,
            'num_layers': 2,
            'output_dim': len(full_dataset.class_to_idx),
            'val_acc': val_acc,
        }, MODEL_SAVE_PATH)
        print(f"  [SAVED] New best model (val_acc={val_acc*100:.2f}%) -> {MODEL_SAVE_PATH}")

print(f"\n[DONE] Training finished. Best validation accuracy: {best_val_acc*100:.2f}%")

[INFO] Loading all data into RAM...
[INFO] Loaded 39101 samples from 310 classes (0 custom samples, Each one is aggregated x1).
[INFO] Training samples: 31280 | Validation samples: 7821
[INFO] Saved idx_to_class mapping -> /content/drive/MyDrive/KArSL_Project/idx_to_class.json
Epoch [1/30] | Train Loss: 2.8165 Acc: 35.79% || Val Loss: 1.5286 Acc: 59.99%
  [SAVED] New best model (val_acc=59.99%) -> /content/drive/MyDrive/KArSL_Project/sign_lstm_model_best.pth
Epoch [2/30] | Train Loss: 1.2540 Acc: 67.17% || Val Loss: 1.0227 Acc: 72.29%
  [SAVED] New best model (val_acc=72.29%) -> /content/drive/MyDrive/KArSL_Project/sign_lstm_model_best.pth
Epoch [3/30] | Train Loss: 0.8726 Acc: 76.47% || Val Loss: 0.7470 Acc: 79.02%
  [SAVED] New best model (val_acc=79.02%) -> /content/drive/MyDrive/KArSL_Project/sign_lstm_model_best.pth
Epoch [4/30] | Train Loss: 0.6357 Acc: 82.17% || Val Loss: 0.5994 Acc: 82.53%
  [SAVED] New best model (val_acc=82.53%) -> /content/drive/MyDrive/KArSL_Project/sign_ls

## 10. Focused Training: Numbers + Letters + Only 5  Words


these cells train a **separate** model that focuses only on:
- Numbers (0001–0031)
- Letters of the alphabet (0032–0070)
- 5 words: **shukran** (thank you), **sadeeq** (friend), **bayt** (house), **bab** (door), and **sareer** (bed).

In [7]:
NUMBER_CLASSES = [f'{i:04d}' for i in range(1, 32)]       # 0001-0031: Numbers
LETTER_CLASSES = [f'{i:04d}' for i in range(32, 71)]      # 0032-0070: The Alphabet
CHOSEN_WORD_CLASSES = ['0293', '0294', '0299', '0302', '0306']  # شكراً، صديق، بيت، باب، سرير

TARGET_CLASS_DIRS = set(NUMBER_CLASSES + LETTER_CLASSES + CHOSEN_WORD_CLASSES)
print(f"[INFO] Target Scope: {len(TARGET_CLASS_DIRS)} Class "
      f"({len(NUMBER_CLASSES)} Number + {len(LETTER_CLASSES)} Letter + {len(CHOSEN_WORD_CLASSES)} Word)")

missing = [c for c in TARGET_CLASS_DIRS if not os.path.isdir(os.path.join(PROCESSED_SAVE_PATH, c))]
if missing:
    print(f"[WARNING] These classes do not exist in processed_landmarks: {sorted(missing)}")
else:
    print("[OK] All required classes are already available no additional extraction is needed..")

[INFO] Target Scope: 75 Class (31 Number + 39 Letter + 5 Word)
[OK] All required classes are already available no additional extraction is needed..


In [8]:
focused_dataset = KArSLInMemoryDataset(PROCESSED_SAVE_PATH, allowed_classes=TARGET_CLASS_DIRS)

idx_to_class_focused = {idx: c_name for c_name, idx in focused_dataset.class_to_idx.items()}
FOCUSED_IDX_TO_CLASS_PATH = f'{BASE_DRIVE_PATH}/idx_to_class_focused.json'
with open(FOCUSED_IDX_TO_CLASS_PATH, 'w', encoding='utf-8') as f:
    json.dump(idx_to_class_focused, f, ensure_ascii=False, indent=2)
print(f"[INFO] Saved -> {FOCUSED_IDX_TO_CLASS_PATH}")

train_size_f = int(0.8 * len(focused_dataset))
val_size_f = len(focused_dataset) - train_size_f
generator_f = torch.Generator().manual_seed(42)
train_subset_f, val_subset_f = random_split(focused_dataset, [train_size_f, val_size_f], generator=generator_f)

train_dataset_f = AugmentedSubset(focused_dataset, train_subset_f.indices, augment=True)
val_dataset_f = AugmentedSubset(focused_dataset, val_subset_f.indices, augment=False)

train_loader_f = DataLoader(train_dataset_f, batch_size=64, shuffle=True, num_workers=0)
val_loader_f = DataLoader(val_dataset_f, batch_size=64, shuffle=False, num_workers=0)

print(f"[INFO] Focused training samples: {len(train_dataset_f)} train | {len(val_dataset_f)} val")

FOCUSED_MODEL_SAVE_PATH = f'{BASE_DRIVE_PATH}/sign_lstm_model_focused.pth'

model_f = SignLSTM(INPUT_DIM, hidden_dim=128, output_dim=len(focused_dataset.class_to_idx)).to(device)
criterion_f = nn.CrossEntropyLoss()
optimizer_f = torch.optim.Adam(model_f.parameters(), lr=0.001)
scheduler_f = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_f, mode='max', factor=0.5, patience=3)

num_epochs_f = 30
best_val_acc_f = 0.0

for epoch in range(num_epochs_f):
    model_f.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    for sequences, labels in train_loader_f:
        sequences, labels = sequences.to(device), labels.to(device)
        outputs = model_f(sequences)
        loss = criterion_f(outputs, labels)

        optimizer_f.zero_grad()
        loss.backward()
        optimizer_f.step()

        train_loss += loss.item() * sequences.size(0)
        train_correct += (outputs.argmax(1) == labels).sum().item()
        train_total += labels.size(0)

    model_f.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for sequences, labels in val_loader_f:
            sequences, labels = sequences.to(device), labels.to(device)
            outputs = model_f(sequences)
            loss = criterion_f(outputs, labels)

            val_loss += loss.item() * sequences.size(0)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += labels.size(0)

    train_acc = train_correct / train_total
    val_acc = val_correct / val_total
    scheduler_f.step(val_acc)

    print(f"[Focused] Epoch [{epoch+1}/{num_epochs_f}] | "
          f"Train Acc: {train_acc*100:.2f}% || Val Acc: {val_acc*100:.2f}%")

    if val_acc > best_val_acc_f:
        best_val_acc_f = val_acc
        torch.save({
            'model_state_dict': model_f.state_dict(),
            'input_dim': INPUT_DIM,
            'hidden_dim': 128,
            'num_layers': 2,
            'output_dim': len(focused_dataset.class_to_idx),
            'val_acc': val_acc,
        }, FOCUSED_MODEL_SAVE_PATH)
        print(f"  [SAVED] New best focused model (val_acc={val_acc*100:.2f}%) -> {FOCUSED_MODEL_SAVE_PATH}")

print(f"\n[DONE] Focused training finished. Best val accuracy: {best_val_acc_f*100:.2f}%")

[INFO] Loading all data into RAM...
[INFO] Loaded 9529 samples from 75 classes (0 custom samples, Each one is aggregated x1).
[INFO] Saved -> /content/drive/MyDrive/KArSL_Project/idx_to_class_focused.json
[INFO] Focused training samples: 7623 train | 1906 val
[Focused] Epoch [1/30] | Train Acc: 9.84% || Val Acc: 15.06%
  [SAVED] New best focused model (val_acc=15.06%) -> /content/drive/MyDrive/KArSL_Project/sign_lstm_model_focused.pth
[Focused] Epoch [2/30] | Train Acc: 21.74% || Val Acc: 25.55%
  [SAVED] New best focused model (val_acc=25.55%) -> /content/drive/MyDrive/KArSL_Project/sign_lstm_model_focused.pth
[Focused] Epoch [3/30] | Train Acc: 32.43% || Val Acc: 40.19%
  [SAVED] New best focused model (val_acc=40.19%) -> /content/drive/MyDrive/KArSL_Project/sign_lstm_model_focused.pth
[Focused] Epoch [4/30] | Train Acc: 41.14% || Val Acc: 48.01%
  [SAVED] New best focused model (val_acc=48.01%) -> /content/drive/MyDrive/KArSL_Project/sign_lstm_model_focused.pth
[Focused] Epoch [5/30